# 8. Tối ưu hoá truy vấn phân tán: Semijoin và loại bỏ phân mảnh (Query Pruning)

**Mục tiêu**

Mô phỏng một CSDL phân tán gồm nhiều trạm bằng Python. Mọi dữ liệu gửi giữa các trạm đều đi qua một đối tượng `Network`
để **đếm số byte truyền qua mạng**, là chi phí chính cần tối ưu trong CSDL phân tán.

| Thí nghiệm | Nội dung |
|---|---|
| 1 | Truy vấn Q′ (tên trưởng phòng của mỗi phòng): so sánh 3 chiến lược, trong đó có Semijoin |
| 2 | Truy vấn Q (phòng ban của mọi nhân viên): khi nào Semijoin **không** hiệu quả |
| 3 | Loại bỏ phân mảnh dựa trên điều kiện bảo vệ (guard condition) |

**Giả thiết theo ví dụ của sách (mục 23.5):**

- Trạm 1 lưu `EMPLOYEE`: 10.000 dòng, mỗi dòng 100 byte, tổng 1.000.000 byte.
- Trạm 2 lưu `DEPARTMENT`: 100 dòng, mỗi dòng 35 byte, tổng 3.500 byte.
- Truy vấn được gửi tại **Trạm 2**, kết quả cần có tại Trạm 2.
- Kích thước thuộc tính: `Ssn`, `Mgr_ssn` 9 byte; `Fname`, `Lname` 15 byte; `Dname` 10 byte; `Dno`, `Dnumber` 4 byte.

Notebook chỉ dùng Python chuẩn, không cần cài thêm thư viện.

## 1. Kích thước thuộc tính, dữ liệu và các trạm

In [1]:
import random

SIZE = {"ssn": 9, "mgr_ssn": 9, "fname": 15, "lname": 15, "dname": 10, "dno": 4, "dnumber": 4}
EMP_ROW_BYTES  = 100      # một dòng EMPLOYEE đầy đủ
DEPT_ROW_BYTES = 35       # một dòng DEPARTMENT đầy đủ

def row_bytes(cols):
    # Kích thước một dòng sau phép chiếu chỉ giữ các cột cols
    return sum(SIZE[c] for c in cols)

# ---- Sinh dữ liệu (cố định seed để ai chạy cũng ra cùng kết quả) ----
random.seed(42)
FIRST = ["John", "Franklin", "Alicia", "Jennifer", "Ramesh", "Joyce", "Ahmad", "James", "Mary", "Linda"]
LAST  = ["Smith", "Wong", "Zelaya", "Wallace", "Narayan", "English", "Jabbar", "Borg", "Brown", "Davis"]

EMPLOYEE = [{"ssn": f"{100000000 + i}", "fname": random.choice(FIRST), "lname": random.choice(LAST),
             "dno": random.randint(1, 100)} for i in range(10_000)]

managers = random.sample(EMPLOYEE, 100)                       # 100 trưởng phòng khác nhau
DEPARTMENT = [{"dnumber": d + 1, "dname": f"Dept_{d + 1:03d}", "mgr_ssn": managers[d]["ssn"]}
              for d in range(100)]

print(f"EMPLOYEE  : {len(EMPLOYEE):,} dòng x {EMP_ROW_BYTES} byte = {len(EMPLOYEE) * EMP_ROW_BYTES:,} byte (Trạm 1)")
print(f"DEPARTMENT: {len(DEPARTMENT):,} dòng x {DEPT_ROW_BYTES} byte = {len(DEPARTMENT) * DEPT_ROW_BYTES:,} byte (Trạm 2)")

EMPLOYEE  : 10,000 dòng x 100 byte = 1,000,000 byte (Trạm 1)
DEPARTMENT: 100 dòng x 35 byte = 3,500 byte (Trạm 2)


## 2. Mạng và các phép toán cơ bản

- `Network.send()` là **cách duy nhất** để dữ liệu đi từ trạm này sang trạm khác. Hàm ghi lại số dòng, số byte của mỗi lần gửi.
- `project()`, `join()`, `semijoin()` là các phép toán đại số quan hệ, được thực hiện **tại một trạm** (không tốn chi phí mạng).

Định nghĩa semijoin: $R \ltimes_{A=B} S = \pi_{R}(R \bowtie_{A=B} S)$, tức là giữ lại các dòng của R có ít nhất một dòng khớp trong S, và chỉ giữ cột của R.

In [2]:
class Network:
    def __init__(self):
        self.log = []

    def send(self, src, dst, what, rows, bytes_per_row):
        b = len(rows) * bytes_per_row
        self.log.append((src, dst, what, len(rows), bytes_per_row, b))
        return [dict(r) for r in rows]        # bản sao dữ liệu đến trạm đích

    def total(self):
        return sum(x[-1] for x in self.log)

    def report(self, title):
        print(f"\n=== {title} ===")
        print(f"{'Từ':<8}{'Đến':<8}{'Dữ liệu gửi':<44}{'Số dòng':>9}{'Byte/dòng':>11}{'Tổng byte':>12}")
        for src, dst, what, n, bpr, b in self.log:
            print(f"{src:<8}{dst:<8}{what:<44}{n:>9,}{bpr:>11}{b:>12,}")
        print(f"{'TỔNG CHI PHÍ TRUYỀN MẠNG':<80}{self.total():>12,} byte")


def project(rows, cols):
    # Phép chiếu, loại trùng
    seen, out = set(), []
    for r in rows:
        key = tuple(r[c] for c in cols)
        if key not in seen:
            seen.add(key)
            out.append({c: r[c] for c in cols})
    return out

def join(R, S, a, b):
    # Hash join R ⋈ (R.a = S.b), thực hiện tại một trạm
    index = {}
    for s in S:
        index.setdefault(s[b], []).append(s)
    return [{**r, **s} for r in R for s in index.get(r[a], [])]

def semijoin(R, S, a, b):
    # R ⋉ S: các dòng của R có ít nhất một dòng khớp trong S
    keys = {s[b] for s in S}
    return [r for r in R if r[a] in keys]

## 3. Thí nghiệm 1: Truy vấn Q′ – tên trưởng phòng của mỗi phòng

```sql
SELECT d.dname, e.fname, e.lname
FROM department d JOIN employee e ON d.mgr_ssn = e.ssn;
```

Kết quả: 100 dòng, mỗi dòng `Dname + Fname + Lname` = 40 byte. Truy vấn gửi tại Trạm 2.

| Chiến lược | Cách làm |
|---|---|
| A. Truyền thẳng EMPLOYEE | Gửi toàn bộ EMPLOYEE từ Trạm 1 sang Trạm 2, nối tại Trạm 2 |
| B. Truyền DEPARTMENT | Gửi toàn bộ DEPARTMENT sang Trạm 1, nối tại Trạm 1, gửi kết quả về Trạm 2 |
| C. Semijoin | (1) Trạm 2 gửi π<sub>Mgr_ssn</sub>(DEPARTMENT) sang Trạm 1; (2) Trạm 1 tính EMPLOYEE ⋉ F, chỉ giữ cột cần, gửi về Trạm 2; (3) Trạm 2 nối với DEPARTMENT |

In [3]:
RESULT_COLS = ["dname", "fname", "lname"]

def q_prime_A():
    net = Network()
    emp_at_2 = net.send("Trạm 1", "Trạm 2", "EMPLOYEE (toàn bộ)", EMPLOYEE, EMP_ROW_BYTES)
    result = project(join(DEPARTMENT, emp_at_2, "mgr_ssn", "ssn"), RESULT_COLS)
    return net, result

def q_prime_B():
    net = Network()
    dept_at_1 = net.send("Trạm 2", "Trạm 1", "DEPARTMENT (toàn bộ)", DEPARTMENT, DEPT_ROW_BYTES)
    res_at_1 = project(join(dept_at_1, EMPLOYEE, "mgr_ssn", "ssn"), RESULT_COLS)
    result = net.send("Trạm 1", "Trạm 2", "Kết quả (Dname, Fname, Lname)", res_at_1, row_bytes(RESULT_COLS))
    return net, result

def q_prime_C():
    net = Network()
    # Bước 1: Trạm 2 chiếu cột khoá nối và gửi sang Trạm 1
    F = project(DEPARTMENT, ["mgr_ssn"])
    F_at_1 = net.send("Trạm 2", "Trạm 1", "F = π Mgr_ssn (DEPARTMENT)", F, row_bytes(["mgr_ssn"]))
    # Bước 2: Trạm 1 lọc EMPLOYEE ⋉ F, chỉ giữ cột cần, gửi về Trạm 2
    R = project(semijoin(EMPLOYEE, F_at_1, "ssn", "mgr_ssn"), ["ssn", "fname", "lname"])
    R_at_2 = net.send("Trạm 1", "Trạm 2", "R = π Ssn,Fname,Lname (EMPLOYEE ⋉ F)", R, row_bytes(["ssn", "fname", "lname"]))
    # Bước 3: Trạm 2 thực hiện phép nối chính thức R ⋈ DEPARTMENT
    result = project(join(DEPARTMENT, R_at_2, "mgr_ssn", "ssn"), RESULT_COLS)
    return net, result

runs = {"A. Truyền thẳng EMPLOYEE": q_prime_A(), "B. Truyền DEPARTMENT": q_prime_B(), "C. Semijoin": q_prime_C()}
for name, (net, result) in runs.items():
    net.report(f"Q′ – {name}  ({len(result)} dòng kết quả)")


=== Q′ – A. Truyền thẳng EMPLOYEE  (100 dòng kết quả) ===
Từ      Đến     Dữ liệu gửi                                   Số dòng  Byte/dòng   Tổng byte
Trạm 1  Trạm 2  EMPLOYEE (toàn bộ)                             10,000        100   1,000,000
TỔNG CHI PHÍ TRUYỀN MẠNG                                                           1,000,000 byte

=== Q′ – B. Truyền DEPARTMENT  (100 dòng kết quả) ===
Từ      Đến     Dữ liệu gửi                                   Số dòng  Byte/dòng   Tổng byte
Trạm 2  Trạm 1  DEPARTMENT (toàn bộ)                              100         35       3,500
Trạm 1  Trạm 2  Kết quả (Dname, Fname, Lname)                     100         40       4,000
TỔNG CHI PHÍ TRUYỀN MẠNG                                                               7,500 byte

=== Q′ – C. Semijoin  (100 dòng kết quả) ===
Từ      Đến     Dữ liệu gửi                                   Số dòng  Byte/dòng   Tổng byte
Trạm 2  Trạm 1  F = π Mgr_ssn (DEPARTMENT)                        100          9      

In [4]:
def summary(runs, title):
    # Kiểm tra các chiến lược cho cùng kết quả, rồi in bảng so sánh
    results = [sorted(map(lambda r: tuple(r.values()), res)) for _, res in runs.values()]
    assert all(r == results[0] for r in results), "Các chiến lược cho kết quả khác nhau!"
    worst = max(net.total() for net, _ in runs.values())
    print(f"{title}  (cả {len(runs)} chiến lược cho cùng {len(results[0])} dòng kết quả)\n")
    print(f"{'Chiến lược':<28}{'Byte truyền':>14}{'So với tệ nhất':>18}")
    for name, (net, _) in runs.items():
        print(f"{name:<28}{net.total():>14,}{(1 - net.total() / worst) * 100:>17.2f}%")

summary(runs, "BẢNG SO SÁNH – TRUY VẤN Q′")

BẢNG SO SÁNH – TRUY VẤN Q′  (cả 3 chiến lược cho cùng 100 dòng kết quả)

Chiến lược                     Byte truyền    So với tệ nhất
A. Truyền thẳng EMPLOYEE         1,000,000             0.00%
B. Truyền DEPARTMENT                 7,500            99.25%
C. Semijoin                          4,800            99.52%


**Nhận xét:**

- Semijoin chỉ truyền **4.800 byte**: 900 byte khoá (100 × 9) và 3.900 byte nhân viên đã lọc (100 × 39), so với 1.000.000 byte khi truyền thẳng EMPLOYEE (giảm 99,52%).
- Tuy nhiên so sánh công bằng phải đặt cạnh **chiến lược tốt nhất không dùng semijoin** là chiến lược B (7.500 byte). So với B, semijoin vẫn giảm thêm khoảng 36%.
- Semijoin hiệu quả ở đây vì điều kiện nối **lọc rất mạnh**: chỉ 100 trên 10.000 nhân viên là trưởng phòng, nên phần lớn EMPLOYEE không phải gửi đi.

## 4. Thí nghiệm 2: Truy vấn Q – phòng ban của mọi nhân viên

```sql
SELECT e.fname, e.lname, d.dname
FROM employee e JOIN department d ON e.dno = d.dnumber;
```

Lần này **mọi** nhân viên đều có phòng ban, nên điều kiện nối gần như không lọc được dòng nào. Kết quả: 10.000 dòng × 40 byte.

In [5]:
def q_A():
    net = Network()
    emp_at_2 = net.send("Trạm 1", "Trạm 2", "EMPLOYEE (toàn bộ)", EMPLOYEE, EMP_ROW_BYTES)
    return net, project(join(emp_at_2, DEPARTMENT, "dno", "dnumber"), ["ssn"] + RESULT_COLS)

def q_B():
    net = Network()
    dept_at_1 = net.send("Trạm 2", "Trạm 1", "DEPARTMENT (toàn bộ)", DEPARTMENT, DEPT_ROW_BYTES)
    res_at_1 = project(join(EMPLOYEE, dept_at_1, "dno", "dnumber"), ["ssn"] + RESULT_COLS)
    # Kết quả gửi về chỉ cần Fname, Lname, Dname (40 byte); ssn giữ lại để phân biệt người trùng tên
    return net, net.send("Trạm 1", "Trạm 2", "Kết quả (Fname, Lname, Dname)", res_at_1, row_bytes(RESULT_COLS))

def q_C():
    net = Network()
    F = project(DEPARTMENT, ["dnumber"])
    F_at_1 = net.send("Trạm 2", "Trạm 1", "F = π Dnumber (DEPARTMENT)", F, row_bytes(["dnumber"]))
    R = project(semijoin(EMPLOYEE, F_at_1, "dno", "dnumber"), ["ssn", "dno", "fname", "lname"])
    R_at_2 = net.send("Trạm 1", "Trạm 2", "R = π Dno,Fname,Lname (EMPLOYEE ⋉ F)", R, row_bytes(["dno", "fname", "lname"]))
    return net, project(join(R_at_2, DEPARTMENT, "dno", "dnumber"), ["ssn"] + RESULT_COLS)

runs_q = {"A. Truyền thẳng EMPLOYEE": q_A(), "B. Truyền DEPARTMENT": q_B(), "C. Semijoin": q_C()}
for name, (net, result) in runs_q.items():
    net.report(f"Q – {name}  ({len(result):,} dòng kết quả)")
print()
summary(runs_q, "BẢNG SO SÁNH – TRUY VẤN Q")


=== Q – A. Truyền thẳng EMPLOYEE  (10,000 dòng kết quả) ===
Từ      Đến     Dữ liệu gửi                                   Số dòng  Byte/dòng   Tổng byte
Trạm 1  Trạm 2  EMPLOYEE (toàn bộ)                             10,000        100   1,000,000
TỔNG CHI PHÍ TRUYỀN MẠNG                                                           1,000,000 byte

=== Q – B. Truyền DEPARTMENT  (10,000 dòng kết quả) ===
Từ      Đến     Dữ liệu gửi                                   Số dòng  Byte/dòng   Tổng byte
Trạm 2  Trạm 1  DEPARTMENT (toàn bộ)                              100         35       3,500
Trạm 1  Trạm 2  Kết quả (Fname, Lname, Dname)                  10,000         40     400,000
TỔNG CHI PHÍ TRUYỀN MẠNG                                                             403,500 byte

=== Q – C. Semijoin  (10,000 dòng kết quả) ===
Từ      Đến     Dữ liệu gửi                                   Số dòng  Byte/dòng   Tổng byte
Trạm 2  Trạm 1  F = π Dnumber (DEPARTMENT)                        100          4

**Nhận xét:** semijoin vẫn phải gửi về gần như toàn bộ 10.000 nhân viên (340.000 byte), vì bước lọc `EMPLOYEE ⋉ F` không loại được dòng nào.
Lợi ích còn lại chủ yếu đến từ **phép chiếu** (chỉ gửi 34 byte thay vì 100 byte mỗi dòng), không phải từ semijoin.

→ Semijoin chỉ đáng dùng khi **độ chọn lọc của phép nối cao** (ít dòng khớp). Bộ tối ưu truy vấn phân tán phải ước lượng điều này trước khi chọn chiến lược.

## 5. Thí nghiệm 3: Loại bỏ phân mảnh dựa trên điều kiện bảo vệ (Query Pruning)

EMPLOYEE được **phân mảnh ngang** và đặt ở 3 trạm. Mỗi phân mảnh có một **điều kiện bảo vệ (guard condition)** mô tả dữ liệu nó chứa:

| Phân mảnh | Điều kiện bảo vệ | Đặt tại |
|---|---|---|
| EMPD5 | `Dno = 5` | Trạm 2 |
| EMPD4 | `Dno = 4` | Trạm 3 |
| EMP_OTHER | `Dno NOT IN (4, 5)` | Trạm 1 |

Truy vấn gửi tại Trạm 2: `SELECT fname, lname FROM employee WHERE dno = 5;`

- **Không loại bỏ phân mảnh:** gửi truy vấn tới cả 3 trạm, mỗi trạm quét phân mảnh của mình rồi gửi kết quả về.
- **Có loại bỏ phân mảnh:** so điều kiện truy vấn với điều kiện bảo vệ. Điều kiện `Dno = 5` mâu thuẫn với guard của EMPD4 và EMP_OTHER, nên chỉ cần truy vấn EMPD5, vốn nằm ngay tại Trạm 2.

In [6]:
FRAGMENTS = [
    {"name": "EMPD5",     "site": "Trạm 2", "guard": "Dno = 5",           "may_contain": lambda v: v == 5},
    {"name": "EMPD4",     "site": "Trạm 3", "guard": "Dno = 4",           "may_contain": lambda v: v == 4},
    {"name": "EMP_OTHER", "site": "Trạm 1", "guard": "Dno NOT IN (4, 5)", "may_contain": lambda v: v not in (4, 5)},
]
for f in FRAGMENTS:
    f["rows"] = [e for e in EMPLOYEE if f["may_contain"](e["dno"])]
    print(f"{f['name']:<10} tại {f['site']}: {len(f['rows']):>5,} dòng  (guard: {f['guard']})")

def query_dno(value, prune, query_site="Trạm 2"):
    net, scanned, sites, result = Network(), 0, [], []
    for f in FRAGMENTS:
        if prune and not f["may_contain"](value):
            print(f"  Bỏ qua {f['name']} ({f['site']}): Dno = {value} mâu thuẫn với guard '{f['guard']}'")
            continue
        sites.append(f["site"])
        if f["site"] != query_site:                                  # gửi truy vấn con tới trạm xa
            sub_sql = f"SELECT fname, lname FROM {f['name']} WHERE dno = {value}"
            net.send(query_site, f["site"], f"Truy vấn con: {sub_sql}"[:43], [{"sql": sub_sql}], len(sub_sql))
        scanned += len(f["rows"])                                    # trạm phải quét phân mảnh
        part = project([e for e in f["rows"] if e["dno"] == value], ["ssn", "fname", "lname"])
        if f["site"] != query_site:                                  # gửi kết quả về trạm truy vấn
            part = net.send(f["site"], query_site, f"Kết quả từ {f['name']}", part, row_bytes(["fname", "lname"]))
        result += part
    return {"Số trạm phải truy vấn": len(sites), "Số dòng phải quét": scanned,
            "Số lần gửi qua mạng": len(net.log), "Byte truyền": net.total(), "Dòng kết quả": len(result)}

print("\n--- Không loại bỏ phân mảnh ---")
no_prune = query_dno(5, prune=False)
print("\n--- Có loại bỏ phân mảnh ---")
with_prune = query_dno(5, prune=True)

print(f"\n{'Chỉ số':<26}{'Không loại bỏ':>16}{'Có loại bỏ':>14}")
for k in no_prune:
    print(f"{k:<26}{no_prune[k]:>16,}{with_prune[k]:>14,}")

EMPD5      tại Trạm 2:    93 dòng  (guard: Dno = 5)
EMPD4      tại Trạm 3:    99 dòng  (guard: Dno = 4)
EMP_OTHER  tại Trạm 1: 9,808 dòng  (guard: Dno NOT IN (4, 5))

--- Không loại bỏ phân mảnh ---

--- Có loại bỏ phân mảnh ---
  Bỏ qua EMPD4 (Trạm 3): Dno = 5 mâu thuẫn với guard 'Dno = 4'
  Bỏ qua EMP_OTHER (Trạm 1): Dno = 5 mâu thuẫn với guard 'Dno NOT IN (4, 5)'

Chỉ số                       Không loại bỏ    Có loại bỏ
Số trạm phải truy vấn                    3             1
Số dòng phải quét                   10,000            93
Số lần gửi qua mạng                      4             0
Byte truyền                             92             0
Dòng kết quả                            93            93


**Nhận xét:** hai cách cho cùng kết quả, nhưng khi dùng điều kiện bảo vệ, bộ tối ưu loại được 2 trên 3 phân mảnh ngay từ bước
**định vị (localization)**, trước khi gửi bất kỳ truy vấn con nào. Truy vấn chỉ chạy tại Trạm 2: không có lần gửi nào qua mạng (không gửi truy vấn con, không chờ kết quả từ trạm xa),
Trạm 1 và Trạm 3 không phải làm việc, và số dòng phải quét giảm từ 10.000 xuống chưa tới 100.
Lưu ý: khi không loại bỏ, hai trạm xa vẫn phải nhận truy vấn con, quét phân mảnh rồi trả về kết quả rỗng; số byte nhỏ
nhưng mỗi lần gửi là một vòng trao đổi qua mạng (độ trễ), và hai trạm đó tốn công quét hàng nghìn dòng vô ích.

## 6. Liên hệ với 4 bước xử lý truy vấn phân tán

| Bước | Thể hiện trong notebook |
|---|---|
| 1. Query Mapping | Câu SQL trên lược đồ toàn cục `EMPLOYEE`, `DEPARTMENT` được chuyển thành biểu thức đại số (`join`, `project`) |
| 2. Localization | Thay `EMPLOYEE` bằng các phân mảnh; loại phân mảnh nhờ điều kiện bảo vệ (thí nghiệm 3) |
| 3. Global Query Optimization | Chọn chiến lược truyền dữ liệu rẻ nhất giữa các trạm: A, B hay Semijoin (thí nghiệm 1, 2) |
| 4. Local Query Optimization | Mỗi trạm tự chọn cách thực hiện phần việc của mình (ở đây dùng hash join trong `join()`) |

## 7. Kết luận

1. Trong CSDL phân tán, chi phí truyền dữ liệu qua mạng là chi phí chính, nên bộ tối ưu tìm cách **giảm số byte phải gửi**.
2. Semijoin gửi cột khoá nối trước để lọc dữ liệu tại trạm nguồn, nhờ đó chỉ gửi những dòng thực sự tham gia phép nối. Với Q′, chi phí giảm từ 1.000.000 byte còn 4.800 byte.
3. Semijoin chỉ hiệu quả khi phép nối có độ chọn lọc cao. Với Q, hầu hết dòng đều khớp nên semijoin gần như không lọc được gì.
4. Loại bỏ phân mảnh dựa trên điều kiện bảo vệ giúp không phải truy vấn những trạm chắc chắn không có dữ liệu thoả mãn.